In [1]:
!pip install pandas numpy scikit-learn nltk


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import nltk
import re

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [4]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

In [5]:
print(fake.shape)
print(true.shape)

(23481, 4)
(21417, 4)


In [6]:
print(fake.columns)
print(true.columns)

Index(['title', 'text', 'subject', 'date'], dtype='str')
Index(['title', 'text', 'subject', 'date'], dtype='str')


In [7]:
fake["label"] = 0
true["label"] = 1

In [8]:
fake.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [9]:
df = pd.concat([fake, true], axis=0)

In [10]:
print(df.shape)

(44898, 5)


In [11]:
df = df.sample(frac=1, random_state=42)

In [12]:
print(df["label"].value_counts())

label
0    23481
1    21417
Name: count, dtype: int64


In [13]:
df.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [14]:
df["content"] = df["title"] + " " + df["text"]

In [15]:
df["content"].head()

22216    Ben Stein Calls Out 9th Circuit Court: Committ...
4436     Trump drops Steve Bannon from National Securit...
1526     Puerto Rico expects U.S. to lift Jones Act shi...
1377      OOPS: Trump Just Accidentally Confirmed He Le...
8995     Donald Trump heads for Scotland to reopen a go...
Name: content, dtype: str

In [16]:
print(df.shape)

print(df["label"].value_counts())

df.isnull().sum()

df["content"].head()

(44898, 6)
label
0    23481
1    21417
Name: count, dtype: int64


22216    Ben Stein Calls Out 9th Circuit Court: Committ...
4436     Trump drops Steve Bannon from National Securit...
1526     Puerto Rico expects U.S. to lift Jones Act shi...
1377      OOPS: Trump Just Accidentally Confirmed He Le...
8995     Donald Trump heads for Scotland to reopen a go...
Name: content, dtype: str

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(df["content"])

y = df["label"]

In [18]:
print(X.shape)

(44898, 122201)


In [19]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7018766 stored elements and shape (44898, 122201)>
  Coords	Values
  (0, 17321)	0.15584531192152343
  (0, 103572)	0.4002939893164815
  (0, 22455)	0.05656642832235487
  (0, 8130)	0.20248711218740748
  (0, 25783)	0.17049607966597258
  (0, 29285)	0.1475985576443362
  (0, 27389)	0.06621930454947782
  (0, 29257)	0.16659431380093412
  (0, 122199)	0.158719270881267
  (0, 28295)	0.13243860909895563
  (0, 4316)	0.13930656963046723
  (0, 24087)	0.12849662614592242
  (0, 118650)	0.13819872949535977
  (0, 96413)	0.03883459335771605
  (0, 92162)	0.1316536975074753
  (0, 87007)	0.07447579388344515
  (0, 83259)	0.16222497395729726
  (0, 113456)	0.0595585825924199
  (0, 52237)	0.07757775471741331
  (0, 41172)	0.10313453312898248
  (0, 12867)	0.08805983488240435
  (0, 111714)	0.12055227258626158
  (0, 99530)	0.05993953250861946
  (0, 42376)	0.10682398716861312
  (0, 42037)	0.158719270881267
  :	:
  (44897, 109375)	0.10247457190839131
  (4489

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(df["content"])

y = df["label"]

print(X.shape)

(44898, 122201)


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [22]:
print(X_train.shape)
print(X_test.shape)

(35918, 122201)
(8980, 122201)


In [23]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [24]:
y_pred = model.predict(X_test)

In [25]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9857461024498887


In [26]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[4634   76]
 [  52 4218]]


In [27]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

In [29]:
sample = true.iloc[0]["title"] + " " + true.iloc[0]["text"]

prediction = model.predict(vectorizer.transform([sample]))

print(prediction)

[1]


In [30]:
sample = fake.iloc[0]["title"] + " " + fake.iloc[0]["text"]

prediction = model.predict(vectorizer.transform([sample]))

print(prediction)

[0]


In [31]:
print(true.iloc[10]["title"])
print()
print(true.iloc[10]["text"])

Jones certified U.S. Senate winner despite Moore challenge

(Reuters) - Alabama officials on Thursday certified Democrat Doug Jones the winner of the state’s U.S. Senate race, after a state judge denied a challenge by Republican Roy Moore, whose campaign was derailed by accusations of sexual misconduct with teenage girls. Jones won the vacant seat by about 22,000 votes, or 1.6 percentage points, election officials said. That made him the first Democrat in a quarter of a century to win a Senate seat in Alabama.  The seat was previously held by Republican Jeff Sessions, who was tapped by U.S. President Donald Trump as attorney general. A state canvassing board composed of Alabama Secretary of State John Merrill, Governor Kay Ivey and Attorney General Steve Marshall certified the election results. Seating Jones will narrow the Republican majority in the Senate to 51 of 100 seats. In a statement, Jones called his victory “a new chapter” and pledged to work with both parties. Moore declined

In [32]:
print(fake.iloc[10]["title"])
print()
print(fake.iloc[10]["text"])

 Papa John’s Founder Retires, Figures Out Racism Is Bad For Business

A centerpiece of Donald Trump s campaign, and now his presidency, has been his white supremacist ways. That is why so many of the public feuds he gets into involve people of color. One of his favorite targets, is, of course, the players in the National Football League who dare to exercise their First Amendment rights by kneeling during the national anthem in protest of racist police brutality. Well, there is one person who has figured out that racism is bad for business, even if it did get the orange overlord elected: The founder of the pizza chain Papa John s.This is a man who has never been on the right side of history on any number of issues, and plus his pizza sucks. But, when he decided to complain about the players protesting, his sales really dropped. Turns out racism doesn t pay, and we all know that corporations are all about the bottom line. Therefore, Papa John Schnatter will no longer be CEO of the hack p